# YOLO26m - Train 2 class (crack / no_crack)

Notebook nay duoc thiet ke cho Kaggle `Run All`.

- Dau vao: dataset detect 5 class (YOLO format).
- Chuyen doi: tat ca box -> class `crack` (0), anh khong co box -> class `no_crack` (1).
- Dau ra: model YOLO26m train 2 phase + file bundle.


In [ ]:
!pip -q install -U ultralytics pyyaml tqdm

import os
import json
import math
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Dependencies ready")


In [ ]:
# ==== Config ====
DATASET_ROOT_HINT = None   # Vi du: '/kaggle/input/ten-dataset'. De None de auto-find
DATASET_HINT_KEYWORD = "crack"
MODEL_NAME = "yolo26m.pt"
IMGSZ = 1024
PHASE1_EPOCHS = 48
PHASE2_EPOCHS = 16
RUN_NAME_BASE = "yolo26m_2class_crack_nocrack_v1"

WORK_ROOT = Path("/kaggle/working/input_dataset_2class")
RUNTIME_YAML = Path("/kaggle/working/data_2class_runtime.yaml")
PROJECT_DIR = Path("/kaggle/working/runs/detect")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_data_yaml(dataset_root_hint=None, keyword='crack'):
    candidates = []
    if dataset_root_hint:
        p = Path(dataset_root_hint)
        if p.is_file() and p.name == "data.yaml":
            return p.resolve()
        if p.is_dir() and (p / "data.yaml").exists():
            return (p / "data.yaml").resolve()

    for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not base.exists():
            continue
        for p in base.rglob("data.yaml"):
            s = p.as_posix().lower()
            score = 0
            if keyword and keyword.lower() in s:
                score += 10
            if "road" in s:
                score += 2
            candidates.append((score, len(s), p.resolve()))

    if not candidates:
        return None

    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][2]


def infer_dataset_root_from_dirs(keyword='crack'):
    candidates = []
    for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not base.exists():
            continue
        for train_images in base.rglob('train/images'):
            root = train_images.parent.parent
            val_name = None
            if (root / 'val' / 'images').exists():
                val_name = 'val'
            elif (root / 'valid' / 'images').exists():
                val_name = 'valid'
            if val_name is None:
                continue
            s = root.as_posix().lower()
            score = 0
            if keyword and keyword.lower() in s:
                score += 10
            if (root / 'test' / 'images').exists():
                score += 2
            if (root / 'train' / 'labels').exists() and (root / val_name / 'labels').exists():
                score += 3
            candidates.append((score, len(s), root.resolve(), val_name))
    if not candidates:
        raise FileNotFoundError('Khong tim thay data.yaml va cung khong suy luan duoc split train/val tu thu muc du lieu')

    candidates.sort(key=lambda x: (-x[0], x[1]))
    _, _, root, val_name = candidates[0]
    return root, val_name

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def resolve_split_path(data_yaml_path, data_cfg, split_key):
    raw = data_cfg.get(split_key)
    if raw is None:
        return None
    p = Path(str(raw))
    if p.is_absolute():
        return p

    root = data_cfg.get("path", None)
    if root:
        root_p = Path(str(root))
        if not root_p.is_absolute():
            root_p = (data_yaml_path.parent / root_p).resolve()
        return (root_p / p).resolve()
    return (data_yaml_path.parent / p).resolve()

data_yaml = find_data_yaml(DATASET_ROOT_HINT, DATASET_HINT_KEYWORD)
if data_yaml is not None:
    cfg = load_yaml(data_yaml)
    val_key = "val" if "val" in cfg else ("valid" if "valid" in cfg else None)
    if val_key is None:
        raise SystemExit("data.yaml khong co split val/valid")
    discover_mode = "from_data_yaml"
else:
    inferred_root, val_key = infer_dataset_root_from_dirs(DATASET_HINT_KEYWORD)
    cfg = {
        "path": str(inferred_root),
        "train": "train/images",
        val_key: f"{val_key}/images",
    }
    if (inferred_root / "test" / "images").exists():
        cfg["test"] = "test/images"
    data_yaml = inferred_root / "data.yaml"
    discover_mode = "from_split_dirs"

split_src = {
    "train": resolve_split_path(data_yaml, cfg, "train"),
    "val": resolve_split_path(data_yaml, cfg, val_key),
    "test": resolve_split_path(data_yaml, cfg, "test") if "test" in cfg else None,
}

for req in ["train", "val"]:
    p = split_src.get(req)
    if p is None or (not p.exists()):
        raise SystemExit(f"Split {req} khong ton tai: {p}")
    n_img = sum(1 for x in p.rglob("*") if x.is_file() and x.suffix.lower() in IMG_EXTS)
    if n_img == 0:
        raise SystemExit(f"Split {req} khong co image: {p}")

print("Dataset detect mode:", discover_mode)
print("Using data.yaml:", data_yaml)
print("Split paths:", json.dumps({k: (str(v) if v else None) for k, v in split_src.items()}, indent=2))


In [ ]:
def ensure_clean_dir(p: Path):
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def list_images(image_dir: Path):
    if image_dir is None or (not image_dir.exists()):
        return []
    return sorted([x for x in image_dir.rglob("*") if x.is_file() and x.suffix.lower() in IMG_EXTS])

def yolo_seg_to_bbox(coords):
    xs = coords[0::2]
    ys = coords[1::2]
    x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
    y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
    w = max(1e-9, x2 - x1)
    h = max(1e-9, y2 - y1)
    cx = x1 + w / 2.0
    cy = y1 + h / 2.0
    return [cx, cy, w, h]

def normalize_bbox(b):
    cx, cy, w, h = [float(v) for v in b]
    cx = min(1.0, max(0.0, cx))
    cy = min(1.0, max(0.0, cy))
    w = min(1.0, max(1e-9, w))
    h = min(1.0, max(1e-9, h))
    return [cx, cy, w, h]

def parse_label_as_crack_boxes(label_path: Path):
    crack_boxes = []
    invalid = 0
    if not label_path.exists():
        return crack_boxes, invalid

    for raw in label_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            invalid += 1
            continue

        try:
            vals = [float(x) for x in parts[1:]]
        except Exception:
            invalid += 1
            continue

        if len(parts) == 5:
            box = normalize_bbox(vals[:4])
            crack_boxes.append(box)
            continue

        # Segment format: cls x1 y1 x2 y2 ... -> convert to bbox
        if len(vals) >= 6 and len(vals) % 2 == 0:
            box = normalize_bbox(yolo_seg_to_bbox(vals))
            crack_boxes.append(box)
        else:
            invalid += 1

    return crack_boxes, invalid

def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
        return
    except Exception:
        pass
    try:
        os.symlink(src, dst)
        return
    except Exception:
        pass
    shutil.copy2(src, dst)

ensure_clean_dir(WORK_ROOT)

global_stats = {}
for split_name in ["train", "val", "test"]:
    src_img_dir = split_src.get(split_name)
    if src_img_dir is None or (not src_img_dir.exists()):
        continue

    src_label_dir = src_img_dir.parent / "labels"
    out_img_dir = WORK_ROOT / split_name / "images"
    out_lbl_dir = WORK_ROOT / split_name / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = list_images(src_img_dir)
    crack_images = 0
    no_crack_images = 0
    crack_boxes = 0
    invalid_lines = 0

    for img_path in tqdm(images, desc=f"convert {split_name}"):
        rel = img_path.relative_to(src_img_dir)
        dst_img = out_img_dir / rel
        link_or_copy(img_path, dst_img)

        src_lbl = src_label_dir / rel.with_suffix(".txt")
        dst_lbl = out_lbl_dir / rel.with_suffix(".txt")
        dst_lbl.parent.mkdir(parents=True, exist_ok=True)

        boxes, bad = parse_label_as_crack_boxes(src_lbl)
        invalid_lines += bad

        if len(boxes) > 0:
            crack_images += 1
            crack_boxes += len(boxes)
            lines = [f"0 {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}" for b in boxes]
            dst_lbl.write_text("\n".join(lines) + "\n", encoding="utf-8")
        else:
            no_crack_images += 1
            # Negative image in detect task must have an empty label file.
            dst_lbl.write_text("", encoding="utf-8")

    global_stats[split_name] = {
        "images": len(images),
        "crack_images": crack_images,
        "no_crack_images": no_crack_images,
        "crack_boxes": crack_boxes,
        "invalid_label_lines": invalid_lines,
    }

runtime = {
    "path": str(WORK_ROOT),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": {0: "crack"},
}

if (WORK_ROOT / "test" / "images").exists():
    runtime["test"] = "test/images"

with open(RUNTIME_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(runtime, f, sort_keys=False, allow_unicode=True)

print("Runtime YAML:", runtime)
print(json.dumps(global_stats, indent=2))

if global_stats.get("train", {}).get("images", 0) == 0:
    raise SystemExit("Split train rong sau convert")
if global_stats.get("val", {}).get("images", 0) == 0:
    raise SystemExit("Split val rong sau convert")
if global_stats.get("train", {}).get("crack_images", 0) == 0:
    raise SystemExit("Train khong co anh crack. Khong the train 2 class")
if global_stats.get("train", {}).get("no_crack_images", 0) == 0:
    raise SystemExit("Train khong co anh no_crack. Khong the train 2 class")
print("Saved:", RUNTIME_YAML)


In [ ]:
import torch

def train_with_fallback(start_model, run_name, epochs, lr0, lrf, batch_candidates, extra_args=None):
    extra_args = extra_args or {}
    last_err = None
    for b in batch_candidates:
        try:
            print(f"\\nStart {run_name} from: {start_model} | batch: {b}")
            model = YOLO(start_model)
            res = model.train(
                data=str(RUNTIME_YAML),
                epochs=int(epochs),
                imgsz=int(IMGSZ),
                batch=int(b),
                optimizer="AdamW",
                lr0=float(lr0),
                lrf=float(lrf),
                cos_lr=True,
                weight_decay=5e-4,
                warmup_epochs=3.0,
                warmup_momentum=0.8,
                warmup_bias_lr=0.1,
                seed=SEED,
                project=str(PROJECT_DIR),
                name=run_name,
                exist_ok=True,
                val=True,
                save=True,
                save_period=10,
                patience=12,
                workers=4,
                device=("0,1" if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else ("0" if torch.cuda.is_available() else "cpu")),
                **extra_args,
            )
            best = Path(res.save_dir) / "weights" / "best.pt"
            return {
                "ok": True,
                "batch": int(b),
                "save_dir": str(res.save_dir),
                "best": str(best),
                "error": None,
            }
        except Exception as e:
            last_err = e
            print(f"[train] batch={b} failed: {e}")
    return {
        "ok": False,
        "batch": None,
        "save_dir": None,
        "best": None,
        "error": str(last_err),
    }

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("CUDA:", torch.cuda.is_available(), "| GPU count:", gpu_count)
for i in range(gpu_count):
    print(f"GPU{i}:", torch.cuda.get_device_name(i))

batch_candidates = [8, 6, 4] if gpu_count >= 2 else [4, 3, 2]
phase2_batches = [max(2, batch_candidates[-1]), 2, 1]
phase2_batches = list(dict.fromkeys(phase2_batches))

phase1 = train_with_fallback(
    start_model=MODEL_NAME,
    run_name=f"{RUN_NAME_BASE}_phase1",
    epochs=PHASE1_EPOCHS,
    lr0=0.002,
    lrf=0.01,
    batch_candidates=batch_candidates,
    extra_args={
        "mosaic": 0.25,
        "mixup": 0.05,
        "degrees": 2.0,
        "translate": 0.08,
        "scale": 0.35,
        "fliplr": 0.5,
        "flipud": 0.0,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "close_mosaic": 10,
        "amp": True,
        "plots": True,
    },
)

if not phase1["ok"]:
    raise SystemExit(f"Phase1 failed: {phase1['error']}")

phase2 = train_with_fallback(
    start_model=phase1["best"],
    run_name=f"{RUN_NAME_BASE}_phase2",
    epochs=PHASE2_EPOCHS,
    lr0=0.00035,
    lrf=0.01,
    batch_candidates=phase2_batches,
    extra_args={
        "mosaic": 0.0,
        "mixup": 0.0,
        "degrees": 0.5,
        "translate": 0.03,
        "scale": 0.20,
        "fliplr": 0.5,
        "flipud": 0.0,
        "hsv_h": 0.01,
        "hsv_s": 0.35,
        "hsv_v": 0.2,
        "close_mosaic": 1,
        "amp": True,
        "plots": True,
    },
)

if not phase2["ok"]:
    print("Phase2 failed, fallback to phase1 best")
    best_for_eval = phase1["best"]
else:
    best_for_eval = phase2["best"]

print("phase1:", phase1)
print("phase2:", phase2)
print("best_for_eval:", best_for_eval)


In [ ]:
def metric_pack(m):
    return {
        "precision": float(getattr(m.box, "mp", 0.0)),
        "recall": float(getattr(m.box, "mr", 0.0)),
        "mAP50": float(getattr(m.box, "map50", 0.0)),
        "mAP50-95": float(getattr(m.box, "map", 0.0)),
    }

best_model = YOLO(str(best_for_eval))
val_res = best_model.val(data=str(RUNTIME_YAML), split="val", imgsz=IMGSZ)
val_metrics = metric_pack(val_res)

test_metrics = None
runtime_loaded = yaml.safe_load(RUNTIME_YAML.read_text(encoding="utf-8"))
if "test" in runtime_loaded:
    test_res = best_model.val(data=str(RUNTIME_YAML), split="test", imgsz=IMGSZ)
    test_metrics = metric_pack(test_res)

report = {
    "model": MODEL_NAME,
    "imgsz": IMGSZ,
    "runtime_yaml": str(RUNTIME_YAML),
    "phase1": phase1,
    "phase2": phase2,
    "best_for_eval": str(best_for_eval),
    "val": val_metrics,
    "test": test_metrics,
}

report_path = Path("/kaggle/working/report_metrics_2class.json")
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(json.dumps(report, indent=2))
print("Saved:", report_path)


In [ ]:
import zipfile

bundle_path = Path(f"/kaggle/working/{RUN_NAME_BASE}_bundle.zip")
paths_to_pack = [
    RUNTIME_YAML,
    Path("/kaggle/working/report_metrics_2class.json"),
]

for p in [phase1.get("save_dir"), phase2.get("save_dir")]:
    if p:
        paths_to_pack.append(Path(p))

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_pack:
        if not item.exists():
            continue
        if item.is_file():
            zf.write(item, arcname=item.name)
        else:
            for fp in item.rglob("*"):
                if fp.is_file():
                    zf.write(fp, arcname=fp.relative_to(item.parent))

print("Bundle saved:", bundle_path)
